In [2]:
import json
import os
import numpy as np
import pandas as pd

In [16]:
from floodlight.io.kinexon import (
    get_meta_data, 
    read_position_data_csv
    )
from floodlight.io.dfl import (
    read_event_data_xml, 
    read_teamsheets_from_mat_info_xml, 
    read_pitch_from_mat_info_xml, 
    read_position_data_xml
    )

from floodlight.models.kinematics import DistanceModel, VelocityModel
from floodlight.models.kinetics import MetabolicPowerModel
from floodlight.models.geometry import CentroidModel

In [8]:
positions_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_04_03_positions_raw_observed_DFL-COM-000001_DFL-MAT-J03WMX.xml" 
metadata_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_02_01_matchinformation_DFL-COM-000001_DFL-MAT-J03WMX.xml"                                  

In [14]:
xy, possession, ballstatus, teamsheet, pitch = read_position_data_xml(positions_filepath, metadata_filepath)                 
                                                                                                          
for key in xy:                                                                                        
    print(key)  # 'firstHalf', 'secondHalf'                                                             
    for team_name, xy_obj in xy[key].items():                                                         
      print(f'{team_name}: {xy_obj.xy.shape}')  # 'Home', 'Away', 'Ball'                              

sampled_data = xy['firstHalf']['Home'].xy                                                             
print(sampled_data)   

firstHalf
Home: (70708, 40)
Away: (70708, 40)
Ball: (70708, 2)
secondHalf
Home: (75259, 40)
Away: (75259, 40)
Ball: (75259, 2)
[[ 6.900e+00  5.120e+00        nan ...  5.180e+00        nan        nan]
 [ 6.820e+00  5.100e+00        nan ...  5.230e+00        nan        nan]
 [ 6.720e+00  5.080e+00        nan ...  5.280e+00        nan        nan]
 ...
 [-1.000e-02 -1.139e+01        nan ... -1.003e+01        nan        nan]
 [ 0.000e+00 -1.147e+01        nan ... -1.010e+01        nan        nan]
 [ 0.000e+00 -1.154e+01        nan ... -1.018e+01        nan        nan]]


In [58]:
DistMod = DistanceModel()
VelMod = VelocityModel()
MetPowMod = MetabolicPowerModel()
CentMod = CentroidModel()

kpi_dict = {}
for half in ["firstHalf", "secondHalf"]:
    for team in ["Home", "Away"]:
        print(f"{half} - {team}")
        DistMod.fit(xy[half][team])
        VelMod.fit(xy[half][team])
        MetPowMod.fit(xy[half][team])
        CentMod.fit(xy[half][team])

        kpi_dict[f'distance_covered_{half}_{team}'] = np.array(DistMod.cumulative_distance_covered())
        kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)
        kpi_dict[f'metabolic_power_{half}_{team}'] = np.array(MetPowMod.metabolic_power())
        kpi_dict[f'centroid_{half}_{team}'] = CentMod.centroid().xy


firstHalf - Home


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


firstHalf - Away


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


secondHalf - Home


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


secondHalf - Away


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


In [ ]:

        if "format" not in parameters:
            raise ValueError("'format' is required for plugin execution.")
        
        logging.error(inputs)
        # ----------------- PARSING
        with inputs["tracking_data"] as input_data: # TrackingData   
            with input_data.open_file() as t_data: # ZipExtFile
                if parameters["format"] == "kinexon":
                    with tempfile.NamedTemporaryFile(mode='w+b', suffix='.csv') as tmp_data:
                        tmp_data.write(t_data.read())
                        # NOTE: returns: List[XY]
                        pos = read_position_data_csv(tmp_data.name, delimiter=parameters["delimiter"])
                        logging.error(type(pos))
                        for p_ in pos: 
                            logging.error(f'{type(p_)} - {p_.xy.shape} - {p_.framerate}')
                            sampled_data = p_.xy
                elif parameters["format"] == "dfl":
                    with tempfile.NamedTemporaryFile(mode='w+b', suffix='.xml') as tmp_data:  
                        tmp_data.write(t_data.read())
                        with inputs["meta_data"] as meta_data:
                            with meta_data.open_file() as m_data:
                                with tempfile.NamedTemporaryFile(mode='w+b', suffix='.xml') as tmp_meta:
                                    tmp_meta.write(m_data.read())
                                    # NOTE: returns: Tuple[
                                        # Dict[str, Dict[str, XY]],
                                        # Dict[str, Code],
                                        # Dict[str, Code],
                                        # Dict[str, Teamsheet],
                                        # Pitch
                                    # ]
                                    data, po, bs, ts, pitch = read_position_data_xml(tmp_data.name, tmp_meta.name)
                                    logging.error(f'{type(data)} - {type(po)} - {type(bs)} - {type(ts)} - {type(pitch)}')
                                    # pID_dict, no_frames, framerate, t_null = get_meta_data(m_data, parameters["delimiter"])
                                    for key in data:
                                        logging.error(key) # 'firstHalf', 'secondHalf'
                                        for pos_data in data[key].items():
                                            logging.error(type(pos_data))
                                            logging.error(pos_data[0]) # 'Home', 'Away', 'Ball'
                                            logging.error(f'{pos_data[1]} - {type(pos_data[1].xy.shape)}') # XY object, XY.xy -> np.NDarray
                                    sampled_data = data['firstHalf']['Home'].xy
                                    logging.error(sampled_data)